[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_01/MFM_ch1_heavy_tails/MFM_ch1_heavy_tails.ipynb)

# MFM_ch1_heavy_tails

Heavy tails, gain/loss asymmetry and aggregational Gaussianity: histograms and QQ-plots of standardised returns with Normal-distribution and Student-t fits, survival functions of gains and losses, 1% tail quantiles across markets, excess kurtosis of non-overlapping h-day returns, and Hill plots of the tail index of absolute returns.

*Modelling Financial Markets (MFM), Chapter 1: Stylised Facts and Returns. Author: Daniel Traian Pele.*

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
Teal     = '#17A2B8'

ASSETS = ['sp500', 'bettr', 'btc', 'eurron', 'gold']
COLORS = {'sp500': MainBlue, 'bettr': IDAred, 'btc': Amber, 'eurron': Forest, 'gold': Purple}
PERIODS = {'sp500': 252, 'bettr': 252, 'btc': 365, 'eurron': 252, 'gold': 252}


def save_fig(name):
    """Save the figure as PDF and transparent PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')

SOURCES = {
    'sp500':  ('GSPC.INDX',    'close', '1990-01-01'),
    'bettr':  ('BETTR.INDX',   'close', '2014-09-01'),
    'btc':    ('BTC-USD.CC',   'close', '2014-09-17'),
    'eurron': ('REF:EUR',      'close', '2005-07-01'),   # from the introduction of the new leu (RON), 1 July 2005
    'gold':   ('XAUUSD.FOREX', 'close', '1990-01-01'),
}

LABELS = {'sp500': 'S&P 500', 'bettr': 'BET-TR (Romania)', 'btc': 'Bitcoin',
          'eurron': 'EUR/RON', 'gold': 'Gold'}

# OTC quotes (gold): partial Saturday/Sunday sessions break the Friday -> Monday return,
# so we keep weekdays only (the same convention as in Chapter 0)
WEEKDAYS_ONLY = {'gold'}

_CACHE = {}


def read_market(symbol):
    """Read the daily price table of one symbol (local copy or GitHub)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    df = pd.read_csv(src, parse_dates=['date']).set_index('date').sort_index()
    df.index.name = 'Date'
    return df


def read_reference_rate(currency='EUR', start='2005-07-01', end='2026-09-18'):
    """Official RON reference rate (BNR annual XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    import re
    import urllib.request
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    df = pd.DataFrame(rows, columns=['Date', 'Close']).drop_duplicates('Date').set_index('Date')
    df.index = pd.to_datetime(df.index)
    _CACHE[key] = df.sort_index().loc[start:end]
    return _CACHE[key]


def load_data(name='sp500'):
    """Daily OHLCV with columns Open/High/Low/Close/Volume (Close = the price used)."""
    symbol, col, start = SOURCES[name]
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start)
    d = read_market(symbol).loc[start:]
    if name in WEEKDAYS_ONLY:
        d = d[d.index.dayofweek < 5]
    out = pd.DataFrame({'Open': d['open'], 'High': d['high'], 'Low': d['low'],
                        'Close': d[col], 'Volume': d['volume']})
    return out[out['Close'] > 0].dropna(subset=['Close'])


def load_close(name):
    """Closing price, used in all charts."""
    return load_data(name)['Close'].dropna()


closes = {a: load_close(a) for a in ASSETS}
rets = {a: np.log(c).diff().dropna() for a, c in closes.items()}
# gold (weekdays only) is annualised with its actual number of observations per year
PERIODS['gold'] = len(rets['gold']) / ((closes['gold'].index[-1] - closes['gold'].index[0]).days / 365.25)
spx_ohlc = load_data('sp500')
for a in ASSETS:
    print(f'{LABELS[a]:18s}', rets[a].index[0].date(), '->', rets[a].index[-1].date(), len(rets[a]), 'obs')

## Helper functions

In [ ]:
HORIZONS = [1, 2, 5, 10, 20, 60]

HILL_FRACS = np.round(np.arange(0.005, 0.1001, 0.0025), 4)

## Analysis

In [ ]:
def fig_hist_qq():
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.2))
    for j, a in enumerate(['sp500', 'btc']):
        r = rets[a]
        z = (r - r.mean()) / r.std()
        ax = axes[0, j]
        ax.hist(z, bins=200, density=True, color=COLORS[a], alpha=0.55, range=(-8, 8))
        x = np.linspace(-8, 8, 600)
        ax.plot(x, stats.norm.pdf(x), color=IDAred, lw=1.0, label='Normal distribution')
        nu, loc, sc = stats.t.fit(z)
        ax.plot(x, stats.t.pdf(x, nu, loc, sc), color=Forest, lw=1.0, label=f'Student-t ($\\nu$={nu:.1f})')
        ax.set_xlim(-6, 6)
        ax.set_title(f'{LABELS[a]}: standardised returns', fontsize=8.5, loc='left')
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=2, frameon=False, fontsize=7)
        ax2 = axes[1, j]
        (osm, osr), _ = stats.probplot(z, dist='norm')
        ax2.scatter(osm, osr, s=3, color=COLORS[a], alpha=0.6)
        lim = [min(osm.min(), osr.min()), max(osm.max(), osr.max())]
        ax2.plot(lim, lim, color=IDAred, lw=0.8)
        ax2.set_xlabel('Normal-distribution quantiles'); ax2.set_ylabel('Empirical quantiles')
        ax2.set_title('QQ-plot vs Normal distribution', fontsize=8.5, loc='left')
    plt.tight_layout(h_pad=2.2)
    save_fig('ch1_hist_qq')


def fig_gain_loss():
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9), gridspec_kw={'width_ratios': [1.25, 1]})
    r = rets['sp500']
    for s, c, lab in [(-r[r < 0], IDAred, 'Losses $-r_t$'), (r[r > 0], Forest, 'Gains $r_t$')]:
        x = np.sort(s.values)[::-1]
        surv = np.arange(1, len(x) + 1) / len(r)
        axes[0].loglog(100 * x, surv, color=c, lw=1.0, label=lab)
    axes[0].set_xlim(0.5, 15)
    axes[0].set_xticks([0.5, 1, 2, 5, 10])
    axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    axes[0].xaxis.set_minor_formatter(plt.NullFormatter())
    axes[0].set_xlabel('Daily move $x$ (%, log scale)'); axes[0].set_ylabel('$P(X > x)$')
    axes[0].set_title('S&P 500 tails: losses are heavier', fontsize=8.5, loc='left')
    axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.3), ncol=2, frameon=False)
    q = pd.DataFrame({LABELS[a].replace(' (Romania)', '').replace(' (BNR)', ''): [-100 * rets[a].quantile(0.01), 100 * rets[a].quantile(0.99)]
                      for a in ASSETS}, index=['1% loss', '99% gain']).T
    x = np.arange(len(q))
    axes[1].bar(x - 0.18, q['1% loss'], 0.36, color=IDAred, label='$-q_{0.01}$')
    axes[1].bar(x + 0.18, q['99% gain'], 0.36, color=Forest, label='$q_{0.99}$')
    axes[1].set_xticks(x, q.index, fontsize=7, rotation=20)
    axes[1].set_ylabel('Daily move (%)')
    axes[1].set_title('1% tails by market', fontsize=8.5, loc='left')
    axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.3), ncol=2, frameon=False, fontsize=7)
    plt.tight_layout()
    save_fig('ch1_gain_loss')
    return q


def aggregation_kurtosis():
    out = {}
    for a in ASSETS:
        r = rets[a]
        ks = []
        for h in HORIZONS:
            m = (len(r) // h) * h                                # complete h-day windows only
            agg = r.iloc[:m].groupby(np.arange(m) // h).sum()   # non-overlapping sums
            ks.append(stats.kurtosis(agg))
        out[a] = ks
    return pd.DataFrame(out, index=HORIZONS)


def fig_aggregation():
    k = aggregation_kurtosis()
    fig, ax = plt.subplots(figsize=(6.8, 3.0))
    for a in ASSETS:
        ax.plot(HORIZONS, k[a], marker='o', ms=3.5, color=COLORS[a], label=LABELS[a])
    ax.axhline(0, color=Navy, ls='--', lw=0.8)
    ax.text(1.05, 0.5, 'Normal distribution', color='black', fontsize=7.5, ha='left')
    ax.set_xscale('log'); ax.set_xticks(HORIZONS, [str(h) for h in HORIZONS])
    ax.set_xlabel('Aggregation horizon (trading days, non-overlapping)')
    ax.set_ylabel('Excess kurtosis')
    ax.set_title('Aggregational Gaussianity: tails thin out as the horizon grows', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=5, y=-0.25)
    plt.tight_layout()
    save_fig('ch1_aggregation_kurtosis')
    k.to_csv(os.path.join(TABLE_DIR, 'ch1_aggregation_kurtosis.csv'), float_format='%.4g')
    return k


def hill_estimator(x, k):
    """alpha_hat(k) = [ (1/k) sum_{i=1..k} ln(X_(i) / X_(k+1)) ]^{-1}, X_(1) >= X_(2) >= ..."""
    x = np.sort(np.asarray(x)[np.asarray(x) > 0])[::-1]
    return 1.0 / np.mean(np.log(x[:k] / x[k]))


def hill_table(frac=0.025):
    """Hill tail index for |r_t|, losses (-r_t) and gains (r_t) at k = frac * n."""
    rows = []
    for a in ASSETS:
        r = rets[a].values
        k = int(frac * len(r))
        rows.append({'asset': LABELS[a], 'n': len(r), 'k': k,
                     'alpha_abs': hill_estimator(np.abs(r), k),
                     'alpha_loss': hill_estimator(-r, k), 'alpha_gain': hill_estimator(r, k),
                     'alpha_abs_min_1to5pct': min(hill_estimator(np.abs(r), int(f * len(r))) for f in (0.01, 0.025, 0.05)),
                     'alpha_abs_max_1to5pct': max(hill_estimator(np.abs(r), int(f * len(r))) for f in (0.01, 0.025, 0.05))})
    t = pd.DataFrame(rows).set_index('asset')
    t.to_csv(os.path.join(TABLE_DIR, 'ch1_hill.csv'), float_format='%.4g')
    return t


def fig_hill():
    fig, ax = plt.subplots(figsize=(6.8, 3.1))
    for a in ASSETS:
        r = np.abs(rets[a].values)
        al = [hill_estimator(r, max(int(f * len(r)), 5)) for f in HILL_FRACS]
        ax.plot(100 * HILL_FRACS, al, color=COLORS[a], lw=1.1, label=LABELS[a])
    ax.axhspan(2, 4, color=BandBlue, alpha=0.5, lw=0)
    ax.axhline(4, color=Navy, ls='--', lw=0.8)
    ax.axhline(2, color=Navy, ls=':', lw=0.8)
    ax.text(9.9, 4.05, r'$\alpha = 4$: kurtosis exists above', fontsize=7, color='black', ha='right', va='bottom')
    ax.text(0.6, 1.95, r'$\alpha = 2$: variance exists above', fontsize=7, color='black', ha='left', va='top')
    ax.set_xlabel('Tail fraction $k/n$ used by the estimator (%)')
    ax.set_ylabel(r'Hill estimate $\hat\alpha$')
    ax.set_ylim(1, 6)
    ax.set_title('Hill plots of $|r_t|$: tail index mostly between 2 and 4', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=5, y=-0.25)
    plt.tight_layout()
    save_fig('ch1_hill_plot')

## Run

In [ ]:
TABLE_DIR = '.'
fig_hist_qq()
print(fig_gain_loss().round(2))
print(fig_aggregation().round(2))
fig_hill()
print(hill_table().round(2))

![ch1_hist_qq](./ch1_hist_qq.png)

Output: `ch1_hist_qq.pdf`

![ch1_gain_loss](./ch1_gain_loss.png)

Output: `ch1_gain_loss.pdf`

![ch1_aggregation_kurtosis](./ch1_aggregation_kurtosis.png)

Output: `ch1_aggregation_kurtosis.pdf`

![ch1_hill_plot](./ch1_hill_plot.png)

Output: `ch1_hill_plot.pdf`